# 🩺 Medical Embedding Fine-Tuning — run on Kaggle (free T4 GPU)

End-to-end on the **complete, real medical datasets** pulled straight from the Hugging Face Hub:
prepare data → collect (query, positive, negative) triplets → MTEB medical baseline →
fine-tune (**3 epochs**) → evaluate → before/after comparison.

**One-time setup:** Kaggle → **Settings**: **Accelerator = GPU T4 ×2**, **Internet = On**.

In [ ]:
# Cell 0 — confirm the GPU is on (should print one or two Tesla T4s)
!nvidia-smi -L

In [ ]:
# Cell 1 — clone the project + install dependencies (~2 min)
REPO_URL = "https://github.com/Om-merkle/AI_CAP_MED_EMBED"  # <-- set your repo URL

!git clone {REPO_URL} project
%cd project
!pip install -q -r requirements.txt

In [ ]:
# Cell 2 — optional Hugging Face connection (higher rate limits + enables Hub push).
# Add a secret named HF_TOKEN under Add-ons → Secrets in the Kaggle notebook editor.
# The datasets are public, so this cell is a safe no-op without a token.
import os
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN loaded from Kaggle secrets ✅")
except Exception:
    print("No HF_TOKEN secret found — continuing anonymously (fine for public datasets).")

In [ ]:
# Cell 3 — pull the COMPLETE datasets from the Hugging Face Hub (~3-5 min):
#   * all 33,955 medical flashcards (medalpaca)
#   * all ~232,000 MedEmbed clinical triplets (abhinand/MedEmbed-training-triplets-v1)
# This overwrites the small starter sample bundled in data/raw/.
!python scripts/download_dataset.py
!wc -l data/raw/*.jsonl

## Path A — one-shot pipeline, 3 epochs (recommended)

Pick a medical domain:
- `nfcorpus` — BeIR medical/nutrition IR, official MTEB task **NFCorpus** (honest before/after)
- `medembed` — the complete 232k MedEmbed clinical triplets with expert negatives (mining skipped)
- `flashcards` — 34k medical Q/A flashcards

`--benchmark` also compares candidate models (MiniLM vs BGE vs the published MedEmbed) on your triplets.

In [ ]:
!python run_pipeline.py --domain nfcorpus --base-model BAAI/bge-small-en-v1.5 \
    --epochs 3 --batch-size 32 --benchmark --run-label kaggle-nfcorpus-3ep

In [ ]:
# Before/after comparison — fine-tuned nDCG@10 should beat the baseline.
import json
print(json.dumps(json.load(open("results/comparison.json")), indent=2))

In [ ]:
# Ranked leaderboard of every run so far (persists in results/leaderboard.csv).
from core import leaderboard
leaderboard.to_dataframe()

## 🏆 Medical benchmark leaderboard (MedEmbed-style)

The full benchmark table from the MedEmbed project: every model evaluated on the same
official medical MTEB tasks, **nDCG@10 + MRR@5 per task**, best score per column in
**bold**, and your fine-tuned model highlighted in green.

Results are cached in `results/med_benchmarks.json` — re-running only evaluates what's new.
`QUICK_TASKS` skips TRECCOVID (~171k docs, by far the slowest); switch to
`med_leaderboard.DEFAULT_TASKS` for the complete 5-task table (~1-2 h extra on a T4).

In [ ]:
from core import med_leaderboard

MODELS = [
    "sentence-transformers/all-MiniLM-L6-v2",
    "BAAI/bge-small-en-v1.5",
    "abhinand/MedEmbed-small-v0.1",
    "models/nfcorpus-bge-small-en-v1.5-ft",   # <- your fine-tuned model (from the run above)
]
TASKS = med_leaderboard.QUICK_TASKS  # or med_leaderboard.DEFAULT_TASKS to add TRECCOVID

med_leaderboard.evaluate(models=MODELS, tasks=TASKS)
med_leaderboard.styled(tasks=TASKS)

In [ ]:
# More experiments — the complete MedEmbed clinical triplets (expert negatives).
# ~232k triplets x 3 epochs is a long T4 run — cap with --sample-size if short on time.
!python run_pipeline.py --domain medembed --epochs 3 --batch-size 32 --run-label kaggle-medembed-3ep

# Or the complete medical flashcards:
# !python run_pipeline.py --domain flashcards --epochs 3 --batch-size 32 --run-label kaggle-flashcards-3ep

The fine-tuned model lands in `models/` and all metrics in `results/` — both downloadable
from the notebook's **Output** tab.

## Path B — optional: FastAPI + Streamlit UI behind a public URL

In [ ]:
# Path B (optional) — run the API + UI behind a public cloudflared URL
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared

import subprocess, time
subprocess.Popen(["uvicorn", "api.main:app", "--host", "0.0.0.0", "--port", "8000"])
time.sleep(5)
subprocess.Popen(["streamlit", "run", "app/streamlit_app.py", "--server.port", "8501", "--server.headless", "true"])
time.sleep(5)
!./cloudflared tunnel --url http://localhost:8501 --no-autoupdate